# Silver Layer — Relational Engineering

Clean, deduplicate, and filter Bronze tables down to the **top 500 enriched movies** from `bronze/enrichment/parquet`.

### 1. SparkSession + read Bronze tables

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_unixtime, split, when, array,
    regexp_extract, regexp_replace, trim, lpad, lower,
)

spark = SparkSession.builder \
    .appName("Silver") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

bronze_ratings    = spark.read.parquet("bronze/ratings")
bronze_movies     = spark.read.parquet("bronze/movies")
bronze_links      = spark.read.parquet("bronze/links")
bronze_tags       = spark.read.parquet("bronze/tags")
bronze_enrichment = spark.read.parquet("bronze/enrichment/parquet")

### 2. Extract the 500 valid movieIds

In [ ]:
top500_ids = bronze_enrichment.select("movieId").distinct()
print(f"Top 500 movieIds: {top500_ids.count()}")

### 3. Clean & filter Ratings

In [ ]:
silver_ratings = (
    bronze_ratings
    .drop("_ingestion_timestamp", "_source_file")
    .join(top500_ids, "movieId", "inner")
    .withColumn("rated_at", from_unixtime(col("timestamp")).cast("timestamp"))
    .drop("timestamp")
    .dropDuplicates(["userId", "movieId", "rated_at"])
    .filter(col("userId").isNotNull() & col("rating").isNotNull())
    .filter((col("rating") >= 0.5) & (col("rating") <= 5.0))
)

print(f"silver_ratings: {silver_ratings.count():,} rows")

### 4. Clean & filter Movies

In [ ]:
silver_movies = (
    bronze_movies
    .drop("_ingestion_timestamp", "_source_file")
    .join(top500_ids, "movieId", "inner")
    .dropDuplicates(["movieId"])
    .withColumn(
        "genres",
        when(col("genres") == "(no genres listed)", array())
        .otherwise(split(col("genres"), "\\|"))
    )
    .withColumn("year", regexp_extract(col("title"), r"\((\d{4})\)\s*$", 1).cast("int"))
    .withColumn("clean_title", trim(regexp_replace(col("title"), r"\s*\(\d{4}\)\s*$", "")))
)

print(f"silver_movies: {silver_movies.count():,} rows")

### 5. Clean & filter Links

In [ ]:
silver_links = (
    bronze_links
    .drop("_ingestion_timestamp", "_source_file")
    .join(top500_ids, "movieId", "inner")
    .dropDuplicates(["movieId"])
    .withColumn("imdbId", lpad(col("imdbId").cast("string"), 7, "0"))
)

print(f"silver_links: {silver_links.count():,} rows")

### 6. Clean & filter Tags

In [ ]:
silver_tags = (
    bronze_tags
    .drop("_ingestion_timestamp", "_source_file")
    .join(top500_ids, "movieId", "inner")
    .withColumn("tagged_at", from_unixtime(col("timestamp")).cast("timestamp"))
    .drop("timestamp")
    .withColumn("tag", trim(lower(col("tag"))))
    .filter(col("tag").isNotNull() & (col("tag") != ""))
    .dropDuplicates(["userId", "movieId", "tag", "tagged_at"])
)

print(f"silver_tags: {silver_tags.count():,} rows")

### 7. Master join: movies with links

In [ ]:
silver_movies_with_links = silver_movies.join(silver_links, "movieId", "left")

print(f"silver_movies_with_links: {silver_movies_with_links.count():,} rows")

### 8. Write Silver tables

In [ ]:
silver_ratings.write.mode("overwrite").parquet("silver/ratings")
silver_movies.write.mode("overwrite").parquet("silver/movies")
silver_links.write.mode("overwrite").parquet("silver/links")
silver_tags.write.mode("overwrite").parquet("silver/tags")
silver_movies_with_links.write.mode("overwrite").parquet("silver/movies_with_links")

print("Silver tables written successfully.")

### 9. Sanity checks

In [ ]:
tables = {
    "ratings":            silver_ratings,
    "movies":             silver_movies,
    "links":              silver_links,
    "tags":               silver_tags,
    "movies_with_links":  silver_movies_with_links,
}

print("=== Row Counts ===")
for name, df in tables.items():
    print(f"  {name}: {df.count():,}")

print("\n=== Schemas ===")
for name, df in tables.items():
    print(f"\n{name}:")
    df.printSchema()

print("=== Samples ===")
for name, df in tables.items():
    print(f"\n{name}:")
    df.show(3, truncate=False)